# Khám phá và Chứng minh Cơ chế Sao chép (Replication) trong NoSQL

Hệ thống phân tán của chúng ta sử dụng **MongoDB Replica Set** bao gồm 3 Node:
- `rs-primary` (Port: 27021): Nút chính, chịu trách nhiệm nhận dữ liệu (Read/Write).
- `rs-secondary` (Port: 27022): Nút phụ, tự động sao chép dữ liệu từ Primary. Chỉ cho phép Đọc (Read-only).
- `rs-arbiter` (Port: 27023): Nút trọng tài, không lưu dữ liệu, chỉ dùng để bỏ phiếu khi Primary bị sập.

## Bước 1: Khởi tạo Kết nối Trực tiếp (Direct Connection)
Bình thường, PyMongo sẽ tự động nhận diện Replica Set và chuyển hướng lệnh về Primary. Để demo một cách rõ ràng nhất đặc tính của từng Node, ta sẽ ép PyMongo kết nối trực tiếp (`directConnection=true`) tới từng cổng riêng biệt.

In [6]:
import pymongo
import time

db_name = 'fahasa_btl2'

# Kết nối trực tiếp vào Primary Node (Port 27021)
client_pri = pymongo.MongoClient('mongodb://127.0.0.1:27021/?directConnection=true', serverSelectionTimeoutMS=2000)

# Kết nối trực tiếp vào Secondary Node (Port 27022)
client_sec = pymongo.MongoClient('mongodb://127.0.0.1:27022/?directConnection=true', serverSelectionTimeoutMS=2000)

print("✅ Đã kết nối thành công tới cả 2 Node!")

✅ Đã kết nối thành công tới cả 2 Node!


## Bước 2: Chứng minh Cơ chế Đồng bộ Dữ liệu (Data Synchronization)
Ta sẽ thêm một cuốn sách mới vào **Máy Primary (27021)**. Ngay sau đó, ta truy vấn bên **Máy Secondary (27022)** để xem dữ liệu có tự động được đồng bộ (Copy) qua hay không.

In [9]:
sach_moi = {
    '_id': 'S9999',
    'TenSach': 'Cẩm Nang Hệ Tán Phân Tán (Bản Đặc Biệt)',
    'TacGia': 'Sinh Viên Fahasa',
    'Gia': 1000000
}

# Xóa dữ liệu cũ nếu chạy lại nhiều lần
client_pri[db_name]['sach'].delete_many({'_id': 'S9999'})

print("--- 1. Bắt đầu Insert vào máy Primary (27021) ---")
client_pri[db_name]['sach'].insert_one(sach_moi)
print("Đã thêm sách thành công vào máy Primary!")

print("\n⏳ Chờ 1 giây để Oplog đồng bộ dữ liệu sang Secondary...")
time.sleep(1)

print("\n--- 2. Truy vấn tại máy Secondary (27022) ---")
sach_sec = client_sec[db_name]['sach'].find_one({'_id': 'S9999'})
if sach_sec:
    print("🎉 KẾT QUẢ: Đã tìm thấy sách ở máy Secondary! Dữ liệu đã tự động được sao chép sang!")
    print(f"   Nội dung: {sach_sec['TenSach']} - Giá: {sach_sec['Gia']}")
else:
    print("❌ Lỗi: Chưa thấy dữ liệu đồng bộ.")

--- 1. Bắt đầu Insert vào máy Primary (27021) ---
Đã thêm sách thành công vào máy Primary!

⏳ Chờ 1 giây để Oplog đồng bộ dữ liệu sang Secondary...

--- 2. Truy vấn tại máy Secondary (27022) ---
🎉 KẾT QUẢ: Đã tìm thấy sách ở máy Secondary! Dữ liệu đã tự động được sao chép sang!
   Nội dung: Cẩm Nang Hệ Tán Phân Tán (Bản Đặc Biệt) - Giá: 1000000


## Bước 3: Chứng minh Đặc tính Read-Only của Secondary
Trong mô hình Replication, **Secondary Node KHÔNG ĐƯỢC PHÉP NHẬN LỆNH GHI**. Việc sửa/xóa/thêm chỉ được thực hiện ở Primary để tránh xung đột dữ liệu.
Ta sẽ thử **cố tình** ghi thẳng một cuốn sách vào máy Secondary xem hệ thống phản ứng ra sao.

In [4]:
sach_loi = {
    '_id': 'S0000',
    'TenSach': 'Sách Này Sẽ Bị Lỗi'
}

print("--- Cố tình Insert trực tiếp vào Secondary (27022) ---")
try:
    client_sec[db_name]['sach'].insert_one(sach_loi)
    print("Ghi thành công")
except Exception as e:
    print("⛔ HỆ THỐNG ĐÃ CHẶN LẠI VÀ BÁO LỖI:")
    print(f"  {e}")
    print("\n=> CHỨNG MINH THÀNH CÔNG: Máy Secondary là Read-Only!")

--- Cố tình Insert trực tiếp vào Secondary (27022) ---
⛔ HỆ THỐNG ĐÃ CHẶN LẠI VÀ BÁO LỖI:
  not primary, full error: {'topologyVersion': {'processId': ObjectId('6a18989729af550675c35a80'), 'counter': 4}, 'ok': 0.0, 'errmsg': 'not primary', 'code': 10107, 'codeName': 'NotWritablePrimary', '$clusterTime': {'clusterTime': Timestamp(1779998969, 1), 'signature': {'hash': b'\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00', 'keyId': 0}}, 'operationTime': Timestamp(1779998969, 1)}

=> CHỨNG MINH THÀNH CÔNG: Máy Secondary là Read-Only!


## Bước 4: Tự động Bầu Cử và Chuyển Đổi (Auto-Failover / Election)
Đây là sức mạnh lớn nhất của Replication nhằm đảm bảo tính **Sẵn sàng cao (High Availability)**. Nếu máy Primary bị sự cố (cháy nổ, rớt mạng), hệ thống sẽ tự động tổ chức bầu cử và thăng cấp một máy Secondary lên làm Primary mới.

**THỰC HÀNH CỰC MẠNH:**
1. Bạn hãy mở Terminal (Màn hình dòng lệnh) lên.
2. Chạy lệnh sau để "giết" máy Primary: `docker stop rs-primary`
3. Đợi khoảng 3-5 giây để quá trình bầu cử (Election) diễn ra.
4. Nhấn chạy ô lệnh bên dưới để xem sự kỳ diệu!

In [5]:
print("--- Kiểm tra lại trạng thái máy Secondary (27022) ---")

is_primary = client_sec.admin.command('ismaster').get('ismaster')

if is_primary:
    print("Máy Secondary đã TỰ ĐỘNG được thăng cấp lên làm PRIMARY MỚI!")
    print("\n--- Tiến hành Insert thử vào máy 27022 (Lúc nãy vừa báo lỗi Read-only) ---")
    try:
        client_sec[db_name]['sach'].insert_one({'_id': 'S8888', 'TenSach': 'Sách Ghi Sau Khi Bầu Cử'})
        print("Ghi thành công!")
    except Exception as e:
        print(f"Lỗi: {e}")
else:
    print("❌ Máy Secondary VẪN CHƯA lên làm Primary.")

--- Kiểm tra lại trạng thái máy Secondary (27022) ---
Máy Secondary đã TỰ ĐỘNG được thăng cấp lên làm PRIMARY MỚI!

--- Tiến hành Insert thử vào máy 27022 (Lúc nãy vừa báo lỗi Read-only) ---
Ghi thành công!
